Connected to .venv01 (3.13.11) (Python 3.13.11)

In [ ]:
# ANALYSIS DLINEAR
# SEARCH WORDS :
# YEARLY_DL_ESTIMATION
# DL_FITTED_TARGET_SPACE_PERFORMANCE
# DL_SCORE_CALIBRATION
# DL_THRESHOLD_CALIBRATION
# DL_THRESHOLD_TRADER_CALIBRATION

import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from pathlib import Path
from arch.utility.exceptions import ConvergenceWarning


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

sys.modules.pop("srcB", None)

In [ ]:
# =============================================================================================================
# ======================================= LOAD BASE OUTPUTS ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_04.pkl","rb") as f:
    outputs_by_year = pickle.load(f)

In [ ]:
# =============================================================================================================
# ======================================= IMPORTS FROM SRC ====================================================
# =============================================================================================================

from srcB.market import *
from srcB.helpers import *

In [ ]:
# =============================================================================================================
# ======================================= READ D_BTC AND DF ===================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

d_btc_raw_by_year = {}
df_raw_by_year = {}

for y in YEARS:
    d_btc_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"btc_underlying_minutely_{y}.parquet")
    df_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"derivative_data_{y}.parquet")

In [ ]:
# =============================================================================================================
# ======================================= SET INDEX TO DATETIME ===============================================
# =============================================================================================================

for y in YEARS:
    d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(pd.to_datetime(d_btc_raw_by_year[y]["date_time"]))
    df_raw_by_year[y] = df_raw_by_year[y].set_index(pd.to_datetime(df_raw_by_year[y]["date_time"]))

In [ ]:
# =============================================================================================================
# ======================================= LOAD X_BY_YEAR ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)

for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2

Xq_by_year = {}

for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]

In [ ]:
# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]

In [ ]:
# =============================================================================================================
# ======================================= BORROW RFR COLUMN ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/ready_data/d_btc_by_year_all.pkl", "rb") as f:
    d_btc_by_year = pickle.load(f)

for y in YEARS:
    d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(d_btc_raw_by_year[y].index, method="ffill")

In [ ]:
# =============================================================================================================
# ======================================= DLINEAR PARAMETERS ==================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters/dl_best_params.pkl", "rb") as f:
    best_dl_params = pickle.load(f)

VAL_FRACTION = 0.20
EPOCHS = 40
PATIENCE = 6
SEED = 42

In [ ]:
# =============================================================================================================
# ======================================= YEARLY_DL_ESTIMATION ================================================
# =============================================================================================================

sys.modules.pop("srcB.dl_model", None)
from srcB.dl_model import *

dl_bundle_by_year = {}
dl_targets_by_year = {}
dl_output_by_year = {}

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index().copy()
    garch_df = pd.concat([outputs_by_year[y][q]["garch"] for q in QUARTERS]).sort_index()
    X_df["garch"] = garch_df.reindex(X_df.index)

    dl_output, dl_bundle, dl_targets = dl_mfiv(
        X_df=X_df,
        option_df=df_raw_by_year[y],
        time_df=d_btc_raw_by_year[y],
        sequence_length=168,
        forward_hours=24,
        delta_target=0.20,
        val_fraction=VAL_FRACTION,
        moving_avg=best_dl_params["moving_avg"],
        individual=best_dl_params["individual"],
        epochs=EPOCHS,
        batch_size=best_dl_params["batch_size"],
        learning_rate=best_dl_params["learning_rate"],
        weight_decay=best_dl_params["weight_decay"],
        patience=PATIENCE,
        seed=SEED
    )

    dl_output_by_year[y] = dl_output
    dl_bundle_by_year[y] = dl_bundle
    dl_targets_by_year[y] = dl_targets

    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        dl_q = dl_output.reindex(q_index)
        outputs_by_year[y][q] = outputs_by_year[y][q].drop(columns=dl_q.columns, errors="ignore")
        outputs_by_year[y][q] = pd.concat([outputs_by_year[y][q], dl_q], axis=1)

In [ ]:
# =============================================================================================================
# ================================ DL_FITTED_TARGET_SPACE_PERFORMANCE =========================================
# =============================================================================================================

target_cols = [
    "target_long_straddle",
    "target_short_straddle",
    "target_long_strangle",
    "target_short_strangle"
]

rows = []

for y in YEARS:
    test = pd.concat([
        dl_output_by_year[y][["dl_action"]],
        dl_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    realized = pd.Series(0.0, index=test.index)

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        mask = test["dl_action"] == action
        realized.loc[mask] = test.loc[mask, f"target_{action}"]

    sharpe = realized.mean() / realized.std()

    rows.append({
        "year": y,
        "n": len(realized),
        "mean_target": realized.mean(),
        "target_sharpe": sharpe,
        "profitable": (realized > 0).mean()
    })

target_sharpe = pd.DataFrame(rows)
print(target_sharpe)

   year     n  mean_target  target_sharpe  profitable
0  2019  2194     0.002138       0.086303    0.301732
1  2020  4093     0.002387       0.081888    0.397752
2  2021  5008     0.002983       0.119448    0.489018


In [ ]:
# =============================================================================================================
# ======================================= DL_SCORE_CALIBRATION ================================================
# =============================================================================================================

rows = []

for y in YEARS:
    test = pd.concat([
        dl_output_by_year[y][["dl_action", "dl_best_score"]],
        dl_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    test = test[test["dl_action"] != "no_trade"].copy()
    test["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = test["dl_action"] == action
        test.loc[m, "target_pnl"] = test.loc[m, f"target_{action}"]

    test["score_bin"] = pd.qcut(
        test["dl_best_score"],
        5,
        labels=False,
        duplicates="drop"
    )

    x = (
        test.groupby("score_bin")
        .agg(
            n=("target_pnl", "size"),
            mean_score=("dl_best_score", "mean"),
            mean_target=("target_pnl", "mean"),
            sharpe=("target_pnl", lambda z: z.mean() / z.std()),
            profitable=("target_pnl", lambda z: (z > 0).mean())
        )
        .reset_index()
    )

    x["year"] = y
    rows.append(x)

score_check = pd.concat(rows, ignore_index=True)

print(score_check[[
    "year", "score_bin", "n", "mean_score",
    "mean_target", "sharpe", "profitable"
]].to_string(index=False))

 year  score_bin   n  mean_score  mean_target    sharpe  profitable
 2019          0 283    0.000491    -0.003450 -0.164652    0.378092
 2019          1 282    0.001552     0.000385  0.016555    0.407801
 2019          2 282    0.002888     0.002584  0.103859    0.500000
 2019          3 282    0.004868     0.008685  0.207413    0.485816
 2019          4 283    0.010807     0.008409  0.233298    0.572438
 2020          0 688    0.001024    -0.000820 -0.047439    0.465116
 2020          1 688    0.002926     0.000055  0.002739    0.462209
 2020          2 687    0.005105     0.000384  0.017278    0.446870
 2020          3 688    0.007943     0.000079  0.003564    0.440407
 2020          4 688    0.019143     0.014502  0.256309    0.552326
 2021          0 793    0.000580    -0.000790 -0.037641    0.529634
 2021          1 793    0.001746     0.002737  0.114299    0.631778
 2021          2 792    0.003218     0.002035  0.079032    0.587121
 2021          3 793    0.005396     0.004100  0

In [ ]:
# =============================================================================================================
# ======================================= DL_THRESHOLD_CALIBRATION ============================================
# =============================================================================================================

tests = {}

for y in YEARS:
    x = pd.concat([
        dl_output_by_year[y][["dl_action", "dl_best_score"]],
        dl_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    x = x[x["dl_action"] != "no_trade"].copy()
    x["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = x["dl_action"] == action
        x.loc[m, "target_pnl"] = x.loc[m, f"target_{action}"]

    tests[y] = x

scores_all = pd.concat([tests[y]["dl_best_score"] for y in YEARS])

rows = []

for q in [0, .20, .40, .50, .60, .70, .80, .85, .90]:

    threshold = scores_all.quantile(q)

    for y in YEARS:
        x = tests[y][tests[y]["dl_best_score"] >= threshold]
        sharpe = x["target_pnl"].mean() / x["target_pnl"].std()

        rows.append({
            "quantile": q,
            "threshold": threshold,
            "year": y,
            "n": len(x),
            "mean_target": x["target_pnl"].mean(),
            "sharpe": sharpe
        })

threshold_check = pd.DataFrame(rows)

print(threshold_check.to_string(index=False))

 quantile    threshold  year    n  mean_target   sharpe
     0.00 5.920592e-07  2019 1412     0.003321 0.107788
     0.00 5.920592e-07  2020 3439     0.002841 0.089391
     0.00 5.920592e-07  2021 3964     0.003768 0.134509
     0.20 1.343884e-03  2019 1036     0.005770 0.172396
     0.20 1.343884e-03  2020 2978     0.003402 0.101444
     0.20 1.343884e-03  2021 3038     0.005038 0.170435
     0.40 2.832198e-03  2019  716     0.007322 0.199111
     0.40 2.832198e-03  2020 2433     0.004366 0.121949
     0.40 2.832198e-03  2021 2140     0.006044 0.194906
     0.50 3.796575e-03  2019  560     0.008512 0.218027
     0.50 3.796575e-03  2020 2116     0.004949 0.131377
     0.50 3.796575e-03  2021 1732     0.007137 0.219547
     0.60 4.900424e-03  2019  423     0.007783 0.220931
     0.60 4.900424e-03  2020 1771     0.005763 0.144674
     0.60 4.900424e-03  2021 1332     0.008224 0.237719
     0.70 6.157296e-03  2019  269     0.007332 0.207762
     0.70 6.157296e-03  2020 1421     0.007181 0

In [ ]:
# =============================================================================================================
# ======================================= DL SCORE DISTRIBUTION ===============================================
# =============================================================================================================

for y in YEARS:
    x = tests[y]["dl_best_score"]

    print(
        y,
        "n:", len(x),
        "min:", x.min(),
        "q25:", x.quantile(.25),
        "q50:", x.quantile(.50),
        "q75:", x.quantile(.75),
        "q90:", x.quantile(.90),
        "max:", x.max()
    )

2019 n: 1412 min: 1.2922101e-06 q25: 0.0012563467607833445 q50: 0.002882821368984878 q75: 0.005388132412917912 q90: 0.008887578360736375 max: 0.051494904
2020 n: 3439 min: 3.2878015e-06 q25: 0.0024185244692489505 q50: 0.005068599246442318 q75: 0.008678833022713661 q90: 0.014252008870244045 max: 0.0966676
2021 n: 3964 min: 5.920592e-07 q25: 0.0014491464826278389 q50: 0.0031608453718945384 q75: 0.005994644248858094 q90: 0.009453538153320552 max: 0.04616535


In [ ]:
# =============================================================================================================
# ======================================= SAVE DL BUNDLES =====================================================
# =============================================================================================================

os.makedirs("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/dl", exist_ok=True)
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/dl/dl_bundle_by_year.pkl", "wb") as f:
    pickle.dump(dl_bundle_by_year, f)

In [ ]:
# =============================================================================================================
# ======================================= PREPARE TRADER INPUT ================================================
# =============================================================================================================

for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["rv"] = Xq_by_year[y][q]["rv"].reindex(
            outputs_by_year[y][q].index
        )
        outputs_by_year[y][q]["MFIV_24"] = Xq_by_year[y][q]["MFIV_24"].reindex(
            outputs_by_year[y][q].index
        )

In [ ]:
# =============================================================================================================
# ======================================= DL TRADER ADAPTER ===================================================
# =============================================================================================================
# DLINEAR signals are not altered;
# it only copies them into column names the existing trader expects.
for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["tcn_action"] = outputs_by_year[y][q]["dl_action"]
        outputs_by_year[y][q]["tcn_best_score"] = outputs_by_year[y][q]["dl_best_score"]

In [ ]:
# =============================================================================================================
# ======================================= CACHE FOR WARMUP ====================================================
# =============================================================================================================

CACHE_YEARS = [2019, 2020, 2021]

trader_cache = {
    y: pd.concat(
        [
            outputs_by_year[yy][q]
            for yy in CACHE_YEARS
            if yy <= y
            for q in QUARTERS
        ]
    ).sort_index()
    for y in CACHE_YEARS
}

In [ ]:
# =============================================================================================================
# ================================ DL_THRESHOLD_TRADER_CALIBRATION ============================================
# =============================================================================================================

sys.modules.pop("srcB.trader_testing_tcn", None)
from srcB.trader_testing_tcn import testing_execute_trading_strategy

THRESHOLDS = [0.0]

for q in [.25, .50, .75, .90]:
    threshold = min(
        tests[y]["dl_best_score"].quantile(q)
        for y in YEARS
    )
    THRESHOLDS.append(float(threshold))

THRESHOLDS = sorted(set(THRESHOLDS))

print("DLinear trader thresholds:", THRESHOLDS)

rows = []

for threshold in THRESHOLDS:
    for y in YEARS:

        market = Market(
            option_df=df_raw_by_year[y],
            time_df=d_btc_raw_by_year[y],
            starting_cash=100_000,
            im_multiplier=0.15,
            mm_buffer_rate=0.05,
        )

        snapshots, greek_tests = testing_execute_trading_strategy(
            bt_year=y,
            bt_quarter=None,
            model="tcn_mfiv",
            outputs_by_year=outputs_by_year,
            market=market,
            trader_cache=trader_cache,
            run_by="year",
            tcn_score_threshold=threshold
        )

        equity = pd.Series(
            [s["equity"] for s in snapshots],
            index=[s["time"] for s in snapshots]
        )

        returns = equity.pct_change().dropna()
        sharpe = returns.mean() / returns.std() * np.sqrt(24 * 365)

        rows.append({
            "threshold": threshold,
            "year": y,
            "trades": len(greek_tests),
            "sharpe": sharpe
        })

threshold_trader_check = pd.DataFrame(rows)

print(threshold_trader_check.to_string(index=False))

DLinear trader thresholds: [0.0, 0.0012563467607833445, 0.002882821368984878, 0.005388132412917912, 0.008887578360736375]
 threshold  year  trades    sharpe
  0.000000  2019     227  0.110757
  0.000000  2020     248 -0.137442
  0.000000  2021     236  0.240590
  0.001256  2019     179  0.321071
  0.001256  2020     241 -0.308674
  0.001256  2021     225  0.600319
  0.002883  2019     164  0.508451
  0.002883  2020     235  0.220529
  0.002883  2021     203  0.772219
  0.005388  2019     108  0.442962
  0.005388  2020     209  0.842157
  0.005388  2021     143  0.879992
  0.008888  2019      59  0.480972
  0.008888  2020     163  0.430978
  0.008888  2021      90  1.069133
